In [2]:
from sae_lens import SAE
import yaml

/Users/plato/code/interpret/v/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
sae, _, _ = SAE.from_pretrained(
    release = "gpt2-small-res-jb", # see other options in sae_lens/pretrained_saes.yaml
    sae_id = "blocks.8.hook_resid_pre", # won't always be a hook point
    device = "cpu"
)   

{'model_name': 'gpt2-small', 'hook_point': 'blocks.8.hook_resid_pre', 'hook_point_layer': 8, 'hook_point_head_index': None, 'dataset_path': 'Skylion007/openwebtext', 'is_dataset_tokenized': False, 'context_size': 128, 'use_cached_activations': False, 'cached_activations_path': 'activations/Skylion007_openwebtext/gpt2-small/blocks.8.hook_resid_pre', 'd_in': 768, 'n_batches_in_buffer': 128, 'total_training_tokens': 300000000, 'store_batch_size': 32, 'device': 'cpu', 'seed': 42, 'dtype': 'float32', 'b_dec_init_method': 'geometric_median', 'expansion_factor': 32, 'from_pretrained_path': None, 'l1_coefficient': 8e-05, 'lr': 0.0004, 'lr_scheduler_name': None, 'lr_warm_up_steps': 5000, 'train_batch_size': 4096, 'use_ghost_grads': False, 'feature_sampling_window': 1000, 'feature_sampling_method': None, 'resample_batches': 1028, 'feature_reinit_scale': 0.2, 'dead_feature_window': 5000, 'dead_feature_estimation_method': 'no_fire', 'dead_feature_threshold': 1e-08, 'log_to_wandb': True, 'wandb_pro

In [4]:
# show all parameters in the sae

for name, param in sae.named_parameters():
    print(name, param.size())

b_enc torch.Size([24576])
W_dec torch.Size([24576, 768])
W_enc torch.Size([768, 24576])
b_dec torch.Size([768])


In [2]:
with open('../app/assets/pretrained_saes.yaml', 'r') as f:
    pretrained_saes = yaml.safe_load(f)

# for k, v in pretrained_saes['SAE_LOOKUP'].items():
#     print(k)
#     print(v.keys())

#     for sae in v['saes']:
#         print(sae['id'])


In [5]:
for k, v in pretrained_saes['SAE_LOOKUP'].items():

    if k == 'gpt2-small-res-jb':
        continue

    for sae in v['saes']:
        print(k, sae['id'])




        sae, _, _ = SAE.from_pretrained(
            release = k, # see other options in sae_lens/pretrained_saes.yaml
            sae_id = sae['id'], # won't always be a hook point
            device = "cpu"
        )

        print(sae.cfg.context_size)

gpt2-small-hook-z-kk blocks.0.hook_z
{'d_in': 768, 'd_sae': 24576, 'dtype': 'float32', 'device': 'cpu', 'model_name': 'gpt2-small', 'hook_name': 'blocks.0.attn.hook_z', 'hook_layer': 0, 'hook_head_index': None, 'activation_fn_str': 'relu', 'apply_b_dec_to_input': True, 'finetuning_scaling_factor': False, 'sae_lens_training_version': None, 'prepend_bos': True, 'dataset_path': 'apollo-research/Skylion007-openwebtext-tokenizer-gpt2', 'context_size': 128, 'normalize_activations': 'none'}
<class 'sae_lens.sae.SAEConfig'>


TypeError: SAEConfig.__init__() missing 1 required positional argument: 'dataset_trust_remote_code'

In [ ]:
def classify_sentiment(model, batch):
    with torch.no_grad():
        outputs = model(input_ids=batch)
    logits = outputs.logits[:, -1, :]
    probabilities = softmax(logits, dim=-1)
    probabilities = probabilities[:, TOKENIZED_LABELS]
    predicted_idxs = torch.argmax(probabilities, dim=-1)
    predicted_sentiments = [LABELS[idx.item()] for idx in predicted_idxs]
    return predicted_sentiments

def classify_sentiment_generate(model, batch):
    outputs = model.generate(input_ids=batch, max_length=batch.shape[1] + 1, num_return_sequences=1)
    decoded_outputs = [tokenizer.decode(output) for output in outputs]
    predicted_sentiments = []
    for decoded_output in decoded_outputs:
        last_word = decoded_output.split()[-1].lower()
        if last_word in LABELS:
            predicted_sentiments.append(last_word)
        else:
            predicted_sentiments.append(classify_sentiment_random(model, [decoded_output])[0])
    return predicted_sentiments

def classify_sentiment_random(model, batch):
    return ['positive' if torch.rand(1) < 0.5 else 'negative' for _ in batch]

In [ ]:
sae.cfg.context_size

128

In [ ]:
sae, _, _ = SAE.from_pretrained(
    release = "gpt2-small-res-jb", # see other options in sae_lens/pretrained_saes.yaml
    sae_id = "blocks.8.hook_resid_pre", # won't always be a hook point
    device = "cpu"
)   